## LLM + Prompting

El objetivo de esta sección es utilizar un LLM para generar resúmenes automáticos a partir de los resultados obtenidos en el análisis exploratorio y modelado de churn y para esto se diseñaron dos prompts distintos con enfoques diferentes para posteriormente comparar los resultados y documentar estrategias para reducir riesgos de alucinación.

#### Prompt 1 — Directo y técnico

Promt: Eres un analista de datos senior. El siguiente es un resumen de resultados de un análisis de churn sobre un dataset de 2,020 usuarios con variables de comportamiento en ventana rolling de 30 días. Variables: age, country, events_last_30d, purchases_last_30d, total_spend_last_30d. Target: churned (9.5% positivo). Se entrenaron Logistic Regression (AUC 0.509, F1 macro 0.42) y Random Forest (AUC 0.491, F1 macro 0.51). La importancia de variables en RF fue: spend (29%), age (27%), events (21%), purchases (15%), country (7%). Se detectaron outliers de dominio (edades negativas/200), outliers extremos en eventos (hasta 900,000) y gastos negativos (-300). Escribe un resumen ejecutivo de 5 párrafos con hallazgos clave, limitaciones y próximos pasos

#### Output generado:

El análisis exploró un dataset de comportamiento de usuarios con el objetivo de predecir churn. La tasa de churn del 9.5% refleja un desbalance típico que requiere estrategias específicas de modelado. Se identificaron tres categorías de outliers relevantes: errores de dominio en edad (valores negativos y mayores de 100), anomalías extremas en eventos que sugieren actividad automatizada o de bots, y gastos negativos que podrían corresponder a devoluciones. Cada tipo fue tratado con una estrategia diferenciada según su naturaleza.
Los modelos entrenados (Regresión Logística y Random Forest) presentaron AUC cercano a 0.5, indicando que las variables disponibles tienen poder predictivo limitado en su forma actual. El gasto total y la edad fueron las variables de mayor importancia, seguidas por el nivel de actividad (eventos). La variable país aportó la menor señal, sugiriendo que el comportamiento de churn es relativamente homogéneo entre los mercados analizados.
La principal limitación identificada es la ausencia de variables contextuales clave: historial de soporte, tipo de plan, canal de adquisición y tendencias temporales. Adicionalmente, la naturaleza rolling del cálculo de features introduce riesgos de data leakage que deben resolverse antes de desplegar el modelo. Se detectaron registros duplicados (mismo user_id con mismos valores) que confirman este riesgo.
Para mejorar la capacidad predictiva, se recomienda: enriquecer las features con variables de tendencia y recency, aplicar validación temporal en lugar de split aleatorio, y explorar modelos de gradiente boosting con calibración de probabilidades. La definición del umbral de clasificación debe alinearse con el costo de negocio de una intervención incorrecta (falso positivo) vs. perder un cliente (falso negativo).
Como siguiente paso inmediato se sugiere una revisión del proceso ETL para resolver duplicados y confirmar la definición temporal del target, seguido de un ciclo de feature engineering de 5–7 días antes de re-entrenar.

#### Prompt 2 — Orientado al negocio, sin jerga técnica

Prompt: "Imagina que eres el jefe de retención de clientes de una empresa de tecnología en Latinoamérica. Tu equipo de datos analizó 2,020 clientes y encontró que el 9.5% abandonó el servicio. Los modelos de predicción aún no son confiables (precisión cercana al azar), pero se sabe que los clientes que gastan menos y tienen menor actividad en la plataforma tienen mayor riesgo. También se encontraron datos sucios: clientes con edades imposibles y niveles de actividad sospechosamente altos (posiblemente bots). Escribe un correo de 3 párrafos para tu VP resumiendo el estado del análisis y pidiendo 7 días más para mejorar el modelo, con justificación de negocio."

#### Output generado:

Estimado/a [VP], quiero compartirte el estado de nuestro proyecto de predicción de churn. Analizamos el comportamiento de 2,020 clientes en los últimos 30 días y encontramos que un 9.5% abandonó el servicio — una tasa manejable pero con potencial de optimización significativo. Identificamos que los clientes con menor gasto y menor actividad en la plataforma son los de mayor riesgo, lo que ya nos da dirección para acciones de retención inmediatas aunque el modelo predictivo aún no esté listo.
Sin embargo, el modelo actual todavía no alcanza la confiabilidad necesaria para activaciones automáticas. Detectamos problemas en la calidad de los datos — incluyendo registros con información imposible y comportamientos anómalos que podrían ser bots — que están afectando la precisión. Necesitamos resolver esto antes de usar el modelo para tomar decisiones reales sobre clientes, ya que actuar sobre predicciones incorrectas tiene un costo: intervenciones innecesarias en clientes que no iban a irse, y recursos desperdiciados.
Te solicito 7 días adicionales para limpiar los datos, incorporar variables clave que actualmente no tenemos (como historial de soporte y tipo de plan) y revalidar el modelo. Con esos cambios, estimamos pasar de una precisión aleatoria al rango de AUC 0.70–0.75, lo que nos permitiría identificar con confianza el 60–70% de los clientes en riesgo antes de que abandonen. Esto se traduce directamente en menor costo de adquisición de nuevos clientes para reemplazar los que se van.

#### Comparación y análisis

El Prompt 2 es mejor para la entrega a stakeholders. El Prompt 1 produce un resumen técnicamente completo y preciso, ideal para un informe interno entre analistas. El Prompt 2 produce un texto accionable, con narrativa de negocio, consecuencias concretas y una solicitud específica — mucho más útil para quien toma decisiones pero no lee métricas de AUC.
La elección del prompt óptimo depende de la audiencia: técnicos → Prompt 1; negocio/liderazgo → Prompt 2. En la práctica, ambos se usarían en distintos momentos del mismo proyecto.

#### Estrategias para evitar alucinaciones

Para reducir el riesgo de respuestas incorrectas o inventadas por parte del LLM, se deben aplicar las siguientes estrategias:

* Grounding explícito: Pasar los datos reales en el prompt (números exactos como AUC, conteos, nombres de variables). El LLM no "inventa" si los datos están presentes.
* Instrucción de restricción: Añadir al prompt: "Usa únicamente los datos proporcionados. Si no tienes información suficiente para responder algo, indícalo explícitamente."
* Chain-of-thought + verificación: Pedir al modelo que liste los hechos que usará antes de redactar, para detectar si introdujo datos externos.
* Validación humana de cifras: Todo número en el output del LLM debe cruzarse contra la fuente original (el código Python en este caso).